# Offline Trace-Driven Optimization (MOPSO + Fuzzy)\nNotebook ini menjalankan optimasi **per skenario CSV secara terpisah** menggunakan modul `offline_trace_mopso_colab.py`.\n\nUrutan step:\n1. Install dependency\n2. Upload file modul + CSV skenario (+ opsional base params)\n3. Set konfigurasi run\n4. Jalankan optimasi batch\n5. Download output JSON\n

In [ ]:
# STEP 1 - Install dependency\n!pip -q install pandas numpy scipy scikit-fuzzy

In [ ]:
# STEP 2 - Upload file dari laptop/local\n# WAJIB upload: offline_trace_mopso_colab.py\n# WAJIB upload: semua file CSV skenario (contoh: trace_normal.csv, trace_spike.csv, trace_ramp.csv)\n# OPSIONAL upload: base_fuzzy_params.json\n\nfrom google.colab import files\nuploaded = files.upload()\nprint('Uploaded files:', list(uploaded.keys()))

In [ ]:
# STEP 3 - Set path & konfigurasi\nfrom pathlib import Path\n\nMODULE_PATH = Path('/content/offline_trace_mopso_colab.py')\nCSV_FILES = [\n    '/content/trace_normal.csv',\n    '/content/trace_spike.csv',\n    '/content/trace_ramp.csv',\n]\n\n# Jika tidak ada file base params, set None\nBASE_PARAMS_JSON = '/content/base_fuzzy_params.json'  # atau None\nif BASE_PARAMS_JSON and not Path(BASE_PARAMS_JSON).exists():\n    BASE_PARAMS_JSON = None\n\nOUT_DIR = '/content/mopso_outputs'\n\nPARTICLES = 30\nITERATIONS = 3000\nSPREAD = 8.0\nSEED = 0\nRUNS = 5\nALLOW_REGRESSION = False\n\nassert MODULE_PATH.exists(), f'Module tidak ditemukan: {MODULE_PATH}'\nfor p in CSV_FILES:\n    assert Path(p).exists(), f'CSV tidak ditemukan: {p}'\n\nprint('Module:', MODULE_PATH)\nprint('CSV files:', CSV_FILES)\nprint('Base params:', BASE_PARAMS_JSON)

In [ ]:
# STEP 4 - Load modul Python\nimport importlib.util\n\nspec = importlib.util.spec_from_file_location('mopso_colab', str(MODULE_PATH))\nmod = importlib.util.module_from_spec(spec)\nspec.loader.exec_module(mod)\n\nprint('Module loaded:', mod.__name__)

In [ ]:
# STEP 5 - Quick sanity check (opsional tapi disarankan)\nbase_params = mod.repair_params(mod.load_base_params(BASE_PARAMS_JSON))\nfirst_ds = mod.load_offline_samples_csv(CSV_FILES[0])\nbase_obj = mod.evaluate_dataset_di_bcu(base_params, first_ds)\n\nprint('Scenario sample   :', first_ds.scenario_name)\nprint('Samples / usable  :', first_ds.sample_count, '/', first_ds.used_samples)\nprint('Base DI           :', round(base_obj.di, 6))\nprint('Base BCU          :', round(base_obj.bcu, 6))\nprint('Base Score        :', round(base_obj.balanced_score, 6))

In [ ]:
# STEP 6 - Jalankan optimasi batch per skenario (isolated)\nresults = mod.run_batch_scenarios(\n    csv_files=CSV_FILES,\n    base_params_path=BASE_PARAMS_JSON,\n    out_dir=OUT_DIR,\n    particles=PARTICLES,\n    iterations=ITERATIONS,\n    spread=SPREAD,\n    seed=SEED,\n    runs=RUNS,\n    allow_regression=ALLOW_REGRESSION,\n)\n\nimport json\nprint(json.dumps(results, indent=2, ensure_ascii=False))

In [ ]:
# STEP 7 - Lihat file output\nfrom pathlib import Path\nout_path = Path(OUT_DIR)\nfor p in sorted(out_path.glob('*')):\n    print(p)

In [ ]:
# STEP 8 - Download semua output JSON\nfrom google.colab import files\nfrom pathlib import Path\n\nfor p in sorted(Path(OUT_DIR).glob('*.json')):\n    files.download(str(p))